# Run a trained ACT policy in the MuJoCo simulator

This notebook starts a MuJoCo simulation, opens its 3D viewer next to your code, downloads a trained
**ACT** policy from the Hugging Face Hub and lets it work in the simulated **conveyor sorting cell**.
No robot and no GPU are needed.

The cell: items ride a belt past an SO-101 arm. The arm has to pick each item and drop it into the right bin:
**red, blue and green items go into the bin of their colour, purple or cracked items into the reject bin.**

| Step | What happens |
|---|---|
| 1 | Start the simulation as its own process (`physicalai-mujoco start`) |
| 2 | Look at it in the browser viewer and at the policy's two cameras |
| 3 | Load the policy with `InferenceModel.from_pretrained` |
| 4 | Run it with `RobotRuntime`, exactly as on a real robot, and watch the score |

The policy was trained with [Physical AI Studio](https://github.com/open-edge-platform/physical-ai-studio) on about 100 episodes recorded
from the simulation's built-in demonstrator, then exported to OpenVINO.

## Setup

In [ ]:
%pip install -q "physicalai[plugin-mujoco,transport,notebooks]"

In [ ]:
import json
import subprocess
import sys
import tempfile
import threading
import time
import urllib.request

from IPython.display import IFrame, Image, display

MODEL_REPO = "iMaxxgx/act-so101-conveyor-sort"  # ACT policy, OpenVINO export
SIM_NAME = "notebook-sim"                       # name the simulation registers under

## 1. Start the simulation

`physicalai-mujoco start` runs the simulation in its own process. With `--status-json` it reports what it is doing as JSON lines and ends with a `ready` event
that lists the ports it picked (`--http-port 0` and `--viser-port 0` mean "any free port"). With `--exit-with-parent` it stops when this
notebook's pipe to it closes, so it cannot be left running by accident.

In [ ]:
sim_log = tempfile.NamedTemporaryFile("w+", prefix="mujoco-sim-", suffix=".log", delete=False)  # the simulation's own log
sim = subprocess.Popen(
    [
        sys.executable, "-m", "physicalai_mujoco_plugin", "start",
        "--profile", "so101", "--scene", "conveyor_sort", "--name", SIM_NAME,
        "--status-json", "--exit-with-parent", "--http-port", "0", "--viser-port", "0",
    ],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=sim_log, text=True,
)

ready = None
for line in sim.stdout:
    event = json.loads(line)
    print(event)
    if event["event"] == "ready":
        ready = event
        break
    if event["event"] == "error":
        raise RuntimeError(f"{event['message']} (log: {sim_log.name})")

http_url, viewer_url = ready["http_url"], ready["viewer_url"]

## 2. Look at it

The simulation serves a 3D viewer (a [viser](https://viser.studio) app). You can orbit the camera, drag items around, change the belt speed or reset the scene
with the panel on the right. Keep it visible while the policy runs.

> Running Jupyter on another machine? `viewer_url` points at *that* machine's `localhost`. Forward the port, or start the simulation with `--viser-host 0.0.0.0`.

In [ ]:
IFrame(viewer_url, width="100%", height=620)

The policy sees the world through two cameras: `overview` and `wrist` (the camera on the gripper). The simulation serves them as ordinary
IP cameras, the same way a real robot's cameras would be.

In [ ]:
def snapshot(camera: str) -> Image:
    with urllib.request.urlopen(f"{http_url}/cameras/{camera}/frame.jpg") as response:
        return Image(data=response.read(), width=420)


for name in ("overview", "wrist"):
    print(name)
    display(snapshot(name))

## 3. Load the policy

`InferenceModel.from_pretrained` downloads the exported policy package (an OpenVINO model plus a `manifest.json` that says which inputs it needs
and how to preprocess them) and compiles it. `device="CPU"` is fine here, the model is small.

In [ ]:
from physicalai.inference import InferenceModel

policy = InferenceModel.from_pretrained(MODEL_REPO, backend="openvino", device="CPU")

## 4. Run the policy

A `RobotRuntime` ties four things together, as on hardware:

- the **robot**: `SharedRobot` attaches to the simulation by its name,
- the **cameras**: the simulation's two MJPEG streams. The names (`gripper`, `overview`) must match the policy's inputs,
- the **action source**: `PolicySource` runs the policy. ACT predicts a chunk of 100 actions; `request_threshold=0.98` asks for a fresh chunk after about every second action, which this policy needs
  (it does not recover from a missed grasp by itself),
- the **loop rate**: 30 Hz.

In [ ]:
from physicalai.capture import IPCamera
from physicalai.robot import SharedRobot
from physicalai.runtime import ChunkedActionQueue, LerpSmoother, PolicySource, RobotRuntime, SyncExecution

cameras = {
    "overview": IPCamera(url=f"{http_url}/cameras/overview/mjpeg", width=640, height=480, fps=30),
    "gripper": IPCamera(url=f"{http_url}/cameras/wrist/mjpeg", width=640, height=480, fps=30),
}
runtime = RobotRuntime(
    robot=SharedRobot(name=SIM_NAME),
    action_source=PolicySource(
        model=policy,
        execution=SyncExecution(request_threshold=0.98),
        action_queue=ChunkedActionQueue(smoother=LerpSmoother()),
    ),
    fps=30.0,
    cameras=cameras,
)
runtime.connect()  # attaches to the simulation and the cameras; stays connected between runs

Now run it. The loop runs in a background thread; this cell shows the overview camera and the score about once a second. Watch the viewer above at the same time.

In [ ]:
def episode_status() -> dict:
    with urllib.request.urlopen(f"{http_url}/health") as response:
        return json.load(response)["episode"]


def run_policy(seconds: float) -> None:
    worker = threading.Thread(target=lambda: runtime.run(duration_s=seconds), daemon=True)
    frame = display(snapshot("overview"), display_id=True)
    text = display("", display_id=True)
    start = time.monotonic()
    worker.start()
    while worker.is_alive():
        status = episode_status()
        frame.update(snapshot("overview"))
        text.update(f"{time.monotonic() - start:4.0f} s | items fed {status['spawned']} | score {status['score']}")
        time.sleep(1.0)


run_policy(60)

The score counts items that ended up in the bin they belong in (`correct`), in a wrong bin (`wrong`) or fell off the belt (`missed`).
Over ten items, a good run mostly scores `correct`.

## 5. Make it harder

The belt speed is a simulation setting: items arrive faster, so the arm has less time. Set it over HTTP, reset the scene, and run again.
The policy was trained at 3 cm/s, so expect more misses as the belt speeds up.

In [ ]:
def post(path: str, body: dict | None = None) -> None:
    request = urllib.request.Request(
        f"{http_url}{path}", data=json.dumps(body or {}).encode(), headers={"content-type": "application/json"}
    )
    urllib.request.urlopen(request).close()


post("/conveyor/belt-speed", {"speed": 0.05})  # metres per second (default 0.03, maximum 0.1)
post("/reset")
run_policy(60)

## 6. Clean up

Disconnect the runtime, then close the pipe to the simulation process, which stops it.

In [ ]:
runtime.disconnect()
sim.stdin.close()
sim.wait(timeout=30)

## Where next

- Start another robot or scene: `physicalai-mujoco profiles` lists the robots, and the [plugin README](../README.md) describes the scenes.
- Record your own demonstrations, train in Physical AI Studio and replace `MODEL_REPO` with your own export.
- Run the same policy from the command line: `physicalai run --config runtime.yaml`.